In [1]:
import os

In [2]:
%pwd

'd:\\projects\\project-9 kidney disease\\kidney-desease-classification_again\\kidney-classification\\research'

In [10]:
import tensorflow as tf

In [5]:
import dagshub
dagshub.init(repo_owner='Nilansh-garg', repo_name='kidney-classification', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\MY LENOVO\anaconda3\envs\kidney_cn\lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d660a408-2461-433a-b309-83c3f2060a21&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6a177b2ff94bafff15279c9e30e6157d3413fec0017280b95c3e1281e06fb6a2




Accessing as Nilansh-garg

Initialized MLflow to track repo "Nilansh-garg/kidney-classification"

Repository Nilansh-garg/kidney-classification initialized!

c:\Users\MY LENOVO\anaconda3\envs\kidney_cn\lib\site-packages\mlflow\utils\requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
from dataclasses import dataclass
from  pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri:str
    params_image_size: int
    params_batch_size: list

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [9]:
class ConfiguraionManager:
    def __init__(self,
                 config_filepath =CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scans-image",
            mlflow_uri = "https://dagshub.com/Nilansh-garg/kidney-classification.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
        )
        return eval_config
     

In [ ]:
from urllib.parse import urlparse
import mlflow.keras
model = tf.keras.models.load_model('artifacts/training/model.h5')

OSError: No file or directory found at artifacts/training/model.h5

In [ ]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        
    def _valid_generator(self):
        
        datagenrator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )
        dataflow_kwargs = dict(
            target_size=(self.config.params_image_size[:-1]),
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenrator_kwargs)
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
        
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        
    def save_score(self):
        scores = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }
        save_json(
            path=Path("scores.json"),
            data=scores
        )
    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_experiment("kidney-disease-classification")
        
        with mlflow.start_run(run_name="evaluation"):
            # Log Hyperparameters
            mlflow.log_params(self.config.all_params)
            
            # Log Metrics
            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })
            
            # Log the actual Model (Optional but highly recommended)
            # This allows you to deploy the model directly from DagsHub/MLflow later
            if self.model:
                mlflow.keras.log_model(self.model, "model", registered_model_name="KidneyDiseaseModel")
            

In [ ]:
try:
    config = ConfiguraionManager()
    eval_config = config.get_validation_config()
    evaluation = Evaluation(config=eval_config)
    evaluation.evaluation()
    evaluation.save_score()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e